In [1]:
try:
    import google.colab
    inColab = True
except ImportError:
    inColab = False
if inColab == True:
    from google.colab import drive
    drive.mount("/content/gdrive")

In [2]:
### ★★★ 수정 포인트 ★★★ 
# 코드의 위치 로 이동
if inColab == True:
    %cd /content/gdrive/MyDrive/Colab Notebooks/ggufs

In [ ]:
import os
from datetime import datetime
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

In [ ]:
# .env 로드 (Colab: 작업 폴더, 로컬: 스크립트 폴더)
load_dotenv(dotenv_path='.env', override=True)

GGUF_SOURCE_REPO  = os.getenv('GGUF_SOURCE_REPO',  'hyokwan/modal_merge_test')
GGUF_LOCAL_DIR    = os.getenv('GGUF_LOCAL_DIR',    '').strip()
GGUF_OUTFILE      = os.getenv('GGUF_OUTFILE',      'model.gguf')
GGUF_OUTTYPE      = os.getenv('GGUF_OUTTYPE',      'f16')
GGUF_HF_REPO      = os.getenv('GGUF_HF_REPO',      '')
HF_TOKEN          = os.getenv('HF_TOKEN',           '')
GGUF_LLAMACPP_DIR = os.getenv('GGUF_LLAMACPP_DIR', './llama.cpp')

# 로컬 다운로드 경로 자동 생성 (비워두면 gguf_{모델명}_{날짜})
if not GGUF_LOCAL_DIR:
    today     = datetime.now().strftime('%Y%m%d')
    modelName = GGUF_SOURCE_REPO.split('/')[-1]
    GGUF_LOCAL_DIR = f'gguf_{modelName}_{today}'

print(f"소스 HF 레포    : {GGUF_SOURCE_REPO}")
print(f"로컬 다운 경로  : {GGUF_LOCAL_DIR}")
print(f"GGUF 파일명     : {GGUF_OUTFILE}")
print(f"출력 타입       : {GGUF_OUTTYPE}")
print(f"업로드 HF 레포  : {GGUF_HF_REPO}")
print(f"llama.cpp 경로  : {GGUF_LLAMACPP_DIR}")

# HF 모델 다운로드
snapshot_download(
    repo_id=GGUF_SOURCE_REPO,
    local_dir=GGUF_LOCAL_DIR,
    revision='main',
    max_workers=4,
    token=HF_TOKEN if HF_TOKEN else None,
)
print(f"\n다운로드 완료: {GGUF_LOCAL_DIR}")

In [5]:
# # 구글드라이브에 이미 있으면 두번 실행할 필요는 없다!
# !git clone https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached protobuf-4.25.9-cp310-abi3-win_amd64.whl.metadata (541 bytes)
  Using cached https://download.pytorch.org/whl/nightly/aiohttp-3.9.5-cp312-cp312-win_amd64.whl (369 kB)
  Using cached pytest-8.3.5-py3-none-any.whl.metadata (7.6 kB)
  Using cached https://download.pytorch.org/whl/nightly/pandas-2.2.3-cp312-cp312-win_amd64.whl (11.5 MB)
  Using cached prometheus_client-0.20.0-py3-none-any.whl.met


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import subprocess

convertScript = os.path.join(GGUF_LLAMACPP_DIR, 'convert_hf_to_gguf.py')
cmd = ['python', convertScript, GGUF_LOCAL_DIR, '--outfile', GGUF_OUTFILE, '--outtype', GGUF_OUTTYPE]
print('실행 명령:', ' '.join(cmd))

result = subprocess.run(cmd, check=True)
print(f"\nGGUF 변환 완료: {GGUF_OUTFILE}")

In [ ]:
from huggingface_hub import HfApi

if not GGUF_HF_REPO:
    raise ValueError("GGUF_HF_REPO가 .env에 설정되지 않았습니다.")

api = HfApi()

# 레포가 없으면 자동 생성
try:
    api.repo_info(repo_id=GGUF_HF_REPO, repo_type='model', token=HF_TOKEN if HF_TOKEN else None)
except Exception:
    api.create_repo(repo_id=GGUF_HF_REPO, repo_type='model', token=HF_TOKEN if HF_TOKEN else None)
    print(f"레포 생성 완료: {GGUF_HF_REPO}")

api.upload_file(
    path_or_fileobj=GGUF_OUTFILE,
    path_in_repo=GGUF_OUTFILE,
    repo_id=GGUF_HF_REPO,
    repo_type='model',
    token=HF_TOKEN if HF_TOKEN else None,
)
print(f"업로드 완료: https://huggingface.co/{GGUF_HF_REPO}")